# EDA

Цель анализа - оценить качество исходного Finance Ecommerce Dirty Dataset перед дальнейшей обработкой.

В ходе EDA последовательно рассматриваются структура и типы данных, полнота и дубликаты, корректность отдельных признаков, выбросы и итоговые метрики качества. Основная задача - оценить качество исходного Finance Ecommerce Dirty Dataset перед дальнейшей обработкой.

In [15]:
from pathlib import Path
import pandas as pd

## 1. Структурная проверка данных

Сначала рассмотрим общую структуру датасета: его размер, пример записей и исходные типы столбцов. Это позволит определить, какие признаки уже имеют подходящий формат, а какие требуют преобразования.

In [16]:
data_path = Path("data/finance_ecommerce_dirty_dataset.csv")

if not data_path.exists():
    data_path = Path("../data/finance_ecommerce_dirty_dataset.csv")

data = pd.read_csv(data_path)

data.head()

,TransactionID,Date,AccountID,AccountName,TransactionType,Amount,Currency,ExchangeRate,Balance,Merchant,...,Subcategory,Country,City,PostalCode,CardNumber,Email,Phone,IsFraud,Notes,CustomerSince
0,TX103251,4/3/2025,AC3973,Arjun Sharma,Debit,47.36,USD,80.6864,-1714968.59,ShopEasy,...,Books,UK,Pune,153621.0,XXXX-XXXX-XXXX-4249,arjun.sharma@hotmail.com,-4.469492e+09,No,repeat purchase,16/09/2018
1,TX101874,10/8/2024,AC8970,Dev Kumar,Debit,159.55,inr,1.0000,-1162428.75,AutoZone,...,Concert,Australia,Delhi,380312.0,XXXX-XXXX-5979,dev.kumar@yahoo.com,5.882006e+09,No,cashback credited,11/7/2022
2,TX103746,9/14/2024,AC3777,Amit Gupta,Debit,2.43,GBP,102.0148,-2085783.34,BigBasket,...,Specialty,USA,Gurugram,NaN,XXXX-XXXX-2243,amit.gupta@example.com,8.832898e+08,No,cashback credited,Aug-21
3,TX105269,12/21/2024,AC9635,Ritu Roy,Refund,78.96,AED,18.1216,-2727243.49,ShopEasy,...,Restaurant,Australia,Mumbai,793747.0,XXXX-XXXX-9279,ritu.roy@hotmail.com,6.845400e+09,No,suspected fraud,2024
4,TX100463,4/14/2024,AC1329,Raj Bose,Refund,42.81,USD,78.4379,-259175.06,LocalMart,...,FastFood,USA,Mumbai,755067.0,NaN,raj.bose@hotmail.com,1.228611e+09,No,cashback credited,3/2/2023


In [17]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6060 entries, 0 to 6059
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TransactionID    6057 non-null   str    
 1   Date             6060 non-null   str    
 2   AccountID        6059 non-null   str    
 3   AccountName      6059 non-null   str    
 4   TransactionType  6058 non-null   str    
 5   Amount           6059 non-null   str    
 6   Currency         6055 non-null   str    
 7   ExchangeRate     6058 non-null   float64
 8   Balance          5988 non-null   str    
 9   Merchant         6058 non-null   str    
 10  MerchantPhone    4647 non-null   float64
 11  MerchantEmail    4222 non-null   str    
 12  Category         6058 non-null   str    
 13  Subcategory      6055 non-null   str    
 14  Country          6058 non-null   str    
 15  City             6058 non-null   str    
 16  PostalCode       5742 non-null   float64
 17  CardNumber       5621 non

### Вывод по разделу 1

Датасет состоит из 6060 строк и 23 столбцов. Часть полей имеет типы, которые не соответствуют их смыслу: `Date` и `CustomerSince` хранятся как строки, `Amount` и `Balance` имеют строковый тип, а `Phone` и `Postalcode` определены как числовые значения.

Это означает, что перед дальнейшей обработкой необходимо отдельно проверить корректность таких полей и возможность их приведения к подходящим типам данных.

## 2. Целостность и полнота данных

### 2.1 Анализ пропусков

In [18]:
data.isna().sum()

TransactionID         3
Date                  0
AccountID             1
AccountName           1
TransactionType       2
Amount                1
Currency              5
ExchangeRate          2
Balance              72
Merchant              2
MerchantPhone      1413
MerchantEmail      1838
Category              2
Subcategory           5
Country               2
City                  2
PostalCode          318
CardNumber          439
Email               259
Phone               328
IsFraud              76
Notes               804
CustomerSince         2
dtype: int64

In [19]:
data.isna().mean()

TransactionID      0.000495
Date               0.000000
AccountID          0.000165
AccountName        0.000165
TransactionType    0.000330
Amount             0.000165
Currency           0.000825
ExchangeRate       0.000330
Balance            0.011881
Merchant           0.000330
MerchantPhone      0.233168
MerchantEmail      0.303300
Category           0.000330
Subcategory        0.000825
Country            0.000330
City               0.000330
PostalCode         0.052475
CardNumber         0.072442
Email              0.042739
Phone              0.054125
IsFraud            0.012541
Notes              0.132673
CustomerSince      0.000330
dtype: float64


Наибольшая доля пропущенных значений наблюдается в `MerchantEmail` (~30%), `MerchantPhone` (~23%) и `Notes` (~13%). В `CardNumber`, `Phone`, `PostalCode` и `Email` доля пропусков составляет ~4–7%, остальные поля заполнены практически полностью.

Следовательно, общая полнота датасета достаточно высокая, но для отдельных полей потребуется отдельная обработка пропусков.

### 2.2. Анализ дубликатов и уникальности идентификатора

In [20]:
data.duplicated().sum()

np.int64(1)

В датасете обнаружен 1 полный дубликат строки.

### Проверка уникальности `TransactionID`

`TransactionID` рассматривается как идентификатор транзакции, поэтому важно проверить, действительно ли его значения уникальны. Сначала проверяется столбец целиком, затем — только заполненные значения, чтобы отделить влияние пропусков от реальных повторов идентификаторов.

In [21]:
data["TransactionID"].is_unique

False

In [22]:
data["TransactionID"].dropna().is_unique

False

После исключения пропусков `TransactionID` всё равно остаётся неуникальным. Значит, причина связана не только с отсутствующими значениями — в датасете действительно есть повторяющиеся идентификаторы.

Далее посчитаем количество повторных появлений и посмотрим, какие именно `TransactionID` встречаются несколько раз.

In [23]:
data["TransactionID"].dropna().duplicated().sum()

np.int64(60)

In [24]:
data.loc[
    data["TransactionID"].notna()
    & data["TransactionID"].duplicated(keep=False),
    "TransactionID"
].value_counts()

TransactionID
TX101095    2
TX102573    2
TX102325    2
TX101246    2
TX104906    2
TX104702    2
TX104977    2
TX103687    2
TX101795    2
TX100971    2
TX101784    2
TX101407    2
TX103653    2
TX100603    2
TX103230    2
TX103823    2
TX102006    2
TX101189    2
TX105743    2
TX103769    2
TX104610    2
TX104935    2
TX104235    2
TX102368    2
TX102832    2
TX101992    2
TX105457    2
TX104981    2
TX102629    2
TX102934    2
TX105443    2
TX100945    2
TX100423    2
TX100417    2
TX100330    2
TX100312    2
TX102344    2
TX105773    2
TX104419    2
TX104845    2
TX104712    2
TX105056    2
TX105059    2
TX105474    2
TX103703    2
TX101587    2
TX100234    2
TX102724    2
TX102765    2
TX103193    2
TX105411    2
TX102752    2
TX103374    2
TX104287    2
TX105343    2
TX105584    2
TX101290    2
TX103431    2
TX101891    2
TX102468    2
Name: count, dtype: int64

`TransactionID` рассматривался как идентификатор транзакции. В столбце есть 3 пропущенных значения, а среди заполненных значений обнаружено 60 дубликатов идентификаторов. Проверка повторяющихся значений показывает, что такие `TransactionID` встречаются по два раза.

### Вывод по разделу 2

Анализ полноты показал, что данные имеют неодинаковую степень заполненности: для каждого столбца были определены количество и доля пропущенных значений.

В датасете обнаружен один полный дубликат строки. Отдельная проверка `TransactionID` показала наличие повторяющихся идентификаторов. Это разные проблемы: полный дубликат означает совпадение всей строки, тогда как одинаковый `TransactionID` может встречаться в записях, которые отличаются по другим признакам.

При дальнейшей обработке необходимо отдельно определить правила работы с пропусками, полным дубликатом и повторяющимися идентификаторами.

## 3. Семантический анализ

### 3.1. Категориальные значения

`IsFraud` проверяется отдельно, так как по смыслу должен содержать ограниченный набор категорий. Анализ позволяет выявить значения, которые не соответствуют ожидаемому формату.

In [25]:
data["IsFraud"].value_counts(dropna=False)

IsFraud
No      5921
NaN       76
Yess      45
Yes       18
Name: count, dtype: int64

В `IsFraud` встречаются значения `No`, `Yes`, `Yess` (45) и пропуски (76). 

`Yess` отличается от ожидаемой категории только написанием и указывает на проблему с единообразием категориальных значений. Перед дальнейшей обработкой такие значения необходимо нормализовать.

### 3.2. Поля дат

`Date` и `CustomerSince` по смыслу являются датами, но исходно хранятся как строки. Проверим, насколько однородны их форматы и можно ли корректно преобразовать значения в datetime.

In [26]:
data["CustomerSince"].value_counts(dropna=False)

CustomerSince
unknown       173
2016          154
2025          149
2022          146
2019          144
             ... 
3/25/2020       1
10/18/2022      1
9/5/2025        1
27/11/2017      1
3/9/2025        1
Name: count, Length: 2417, dtype: int64

In [27]:
customer_since_converted = pd.to_datetime(
    data["CustomerSince"],
    errors="coerce",
    format="mixed"
)

data.loc[
    customer_since_converted.isna(),
    "CustomerSince"
].value_counts(dropna=False)

CustomerSince
unknown    173
NaN          2
Name: count, dtype: int64

Столбец `CustomerSince` содержит даты в нескольких форматах, а также значение `unknown`. При преобразовании в datetime с использованием `format="mixed"` большинство значений распознаются корректно. Не преобразуются 173 значения `unknown`, 2 значения являются пропусками.

In [28]:
date_converted = pd.to_datetime(
    data["Date"],
    errors="coerce",
    format="%m/%d/%Y"
)

data.loc[
    date_converted.isna(),
    "Date"
].value_counts(dropna=False)

Series([], Name: count, dtype: int64)

Столбец `Date` хранится в строковом формате, однако все значения соответствуют формату месяц/день/год (%m/%d/%Y). При преобразовании в datetime некорректных значений не обнаружено. Следовательно, проблема столбца заключается только в исходном типе данных, а дополнительная очистка значений не требуется.

### 3.3. Числовые признаки

`Amount`, `Balance` и `ExchangeRate` связаны с количественными характеристиками транзакций. Для них важно проверить возможность числового преобразования и разумность диапазона значений.

In [29]:

amount_numeric = pd.to_numeric(data["Amount"], errors="coerce")

data.loc[
    amount_numeric.isna(),
    "Amount"
].value_counts(dropna=False).head(10)

Amount
2.50x        3
$73.66       2
33.16x       2
INR 14.68    2
Rs 63.57     2
24.92x       2
INR 16.83    2
INR 6.62     2
USD 27.19    2
USD 28.08    2
Name: count, dtype: int64

In [30]:
balance_numeric = pd.to_numeric(data["Balance"], errors="coerce")

data.loc[
    balance_numeric.isna(),
    "Balance"
].value_counts(dropna=False).head(10)

Balance
NaN               72
?-554,639.12       2
?-595,131.99       2
?-1,714,295.08     2
?-176,002.59       2
?-175,307.66       2
?-182,672.26       2
?-1,617,269.64     2
?-2,773,860.42     2
?-1,849,660.56     2
Name: count, dtype: int64

В `Amount` и `Balance` встречаются значения, которые нельзя напрямую преобразовать в числовой тип. 

Оба признака по смыслу являются числовыми, но часть исходных значений содержит валютные обозначения, разделители и дополнительные символы. Именно поэтому pandas изначально воспринимает эти столбцы как строки.

Такие записи нельзя использовать в числовых расчётах без предварительной обработки. При этом важно отличать реальные пропуски от значений, которые стали пропусками только из-за невозможности прямого преобразования.

In [31]:
data["ExchangeRate"].describe()

count    6058.000000
mean       46.319508
std        40.254595
min         1.000000
25%         1.000000
50%        72.091800
75%        80.274425
max       109.968400
Name: ExchangeRate, dtype: float64

`ExchangeRate` уже имеет числовой тип. Минимальное значение равно 1, максимальное — около 110, отрицательных и нулевых значений не обнаружено. Явных семантических аномалий в столбце не выявлено.

### 3.4. Идентификаторы и коды

`MerchantPhone`, `Phone` и `PostalCode` технически были определены как числовые поля, хотя по смыслу являются идентификаторами и кодами. 

In [32]:
data["MerchantPhone"].value_counts(dropna=False).head(15)

MerchantPhone
 NaN             1413
 9.590231e+08       6
-7.122014e+08       5
 5.669108e+09       5
 1.689022e+09       5
 6.655838e+07       5
 3.635479e+09       5
 9.473747e+09       5
-1.529532e+09       5
-7.080457e+09       5
 2.144366e+09       5
 3.991187e+09       4
-5.256654e+09       4
 8.595112e+08       4
-7.940931e+08       4
Name: count, dtype: int64

In [33]:
data["PostalCode"].value_counts(dropna=False).head(15)

PostalCode
NaN         318
851.0         3
880938.0      2
292.0         2
366839.0      2
767751.0      2
460.0         2
676348.0      2
331559.0      2
958281.0      2
221228.0      2
791338.0      2
905570.0      2
733.0         2
202809.0      2
Name: count, dtype: int64

In [34]:
data["Phone"].value_counts(dropna=False).head(15)

Phone
 NaN             328
 8.885465e+09      2
 9.669522e+08      2
 1.051015e+09      2
 5.840069e+09      2
 9.167525e+09      2
-9.262595e+09      2
 5.628702e+09      2
 6.495085e+09      2
 1.599957e+09      2
 3.604458e+08      2
-2.129881e+09      2
 8.618752e+09      2
 3.703928e+09      2
 5.488630e+09      2
Name: count, dtype: int64


`MerchantPhone`, `PostalCode` и `Phone` используются как идентификаторы, их логичнее хранить как строки, а не как числовые значения. Из-за числового типа значения отображаются как числа с плавающей точкой или в экспоненциальной форме. В телефонных полях также встречаются отрицательные значения, которые не соответствуют смыслу признака и являются аномальными. 

### Вывод по разделу 3

Проверка показала несколько проблем: некорректные категории, разные форматы дат, лишние символы в числовых значениях и неподходящие типы данных для некоторых идентификаторов. Требуется дополнительная обработка.

При этом `Date` и `ExchangeRate` не содержат явных проблем и требуют минимальной обработки.

## 4. Поиск выбросов и аномалий

### 4.1. Проверка столбца `Amount`

In [35]:
amount_numeric.describe()

count    4837.000000
mean       61.425200
std       135.533152
min      -598.250000
25%        12.600000
50%        30.920000
75%        68.710000
max      3683.010000
Name: Amount, dtype: float64

Для `Amount` уже по описательной статистике виден заметный разброс значений: среднее выше медианы, а максимальное значение существенно превышает верхний квартиль. Поэтому дополнительно проверим признак на наличие статистических выбросов методом IQR.

In [36]:
q1 = amount_numeric.quantile(0.25)
q3 = amount_numeric.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

lower_bound, upper_bound

(np.float64(-71.565), np.float64(152.875))

Для `Amount` границы IQR составили от -71.565 до 152.875. Значения за пределами этого диапазона будем рассматривать как потенциальные выбросы и далее посчитаем их количество.

In [37]:
amount_outliers = amount_numeric[
    (amount_numeric < lower_bound) | (amount_numeric > upper_bound)
]

amount_outliers.count()

np.int64(519)

Для `Amount` обнаружено 519 значений за границами IQR. Это показывает, что распределение содержит заметное количество экстремальных наблюдений и такие записи требуют отдельной проверки перед возможным удалением или корректировкой.

### 4.2. Проверка столбца `Balance`

In [38]:
balance_numeric.describe()

count    4.825000e+03
mean    -1.630977e+06
std      8.733530e+05
min     -2.936329e+06
25%     -2.463455e+06
50%     -1.641955e+06
75%     -9.872448e+05
max      9.969950e+03
Name: Balance, dtype: float64

`Balance` также имеет широкий диапазон значений, однако сама центральная часть распределения значительно более растянута, чем у `Amount`. Поэтому границы IQR для этого признака получаются намного шире.

In [39]:
q1_balance = balance_numeric.quantile(0.25)
q3_balance = balance_numeric.quantile(0.75)
iqr_balance = q3_balance - q1_balance

lower_bound_balance = q1_balance - 1.5 * iqr_balance
upper_bound_balance = q3_balance + 1.5 * iqr_balance

lower_bound_balance, upper_bound_balance

(np.float64(-4677770.105), np.float64(1227070.4150000003))

Для `Balance` диапазон IQR получился значительно шире: от примерно -4.68 млн до 1.23 млн. Проверим, есть ли значения за этими границами.

In [40]:
balance_outliers = balance_numeric[
    (balance_numeric < lower_bound_balance)
    | (balance_numeric > upper_bound_balance)
]

balance_outliers.count()

np.int64(0)

По методу IQR для `Balance` выбросы не обнаружены. Это не означает, что поле полностью лишено проблем: ранее в нём уже были найдены некорректно оформленные исходные значения. Здесь речь только об отсутствии статистически экстремальных значений среди успешно преобразованных чисел.

### Вывод по разделу 4

Для поиска выбросов были выбраны `Amount` и `Balance`, так как это основные количественные признаки датасета.

Для `Amount` методом IQR были определены границы от -71.565 до 152.875. За их пределами находится 519 значений, которые можно считать потенциальными выбросами.

Для `Balance` границы получились значительно шире из-за большого разброса значений, поэтому выбросов по методу IQR не обнаружено.

При этом результат IQR показывает только статистически необычные значения и сам по себе не означает, что они являются ошибками данных.

## 5. Метрики качества данных

### 5.1. Полнота и дубликаты

In [41]:
completeness = data.notna().mean().mean()

completeness

np.float64(0.9599870856650882)

In [42]:
duplication_rate = data.duplicated().mean()

duplication_rate

np.float64(0.00016501650165016502)

Completeness ≈ 0.960 - в среднем заполнено около 96% ячеек датасета.

Duplication Rate ≈ 0.000165 - доля полных дубликатов около 0.0165%.

Полученные значения показывают, что на общем уровне датасет достаточно полный: заполнено около 96% ячеек. Полные дубликаты практически отсутствуют, поэтому дублирование строк не является основной проблемой качества данных.

### 5.2. Валидность

In [43]:
amount_validity = amount_numeric.notna().mean()

amount_validity

np.float64(0.7981848184818482)

In [44]:
balance_validity = balance_numeric.notna().mean()

balance_validity

np.float64(0.7962046204620462)

Validity для Balance ≈ 0.796 - около 80% значений валидны как числа.

Validity для Amount ≈ 0.798 - около 80% значений валидны как числа.

При этом высокая общая полнота не означает, что все заполненные значения корректны. Только около 80% записей в Amount и Balance удаётся напрямую интерпретировать как числа. Остальные значения требуют очистки перед использованием в расчётах.

### 5.3. Процент аномальных выбросов

In [45]:
amount_outlier_ratio = amount_outliers.count() / amount_numeric.notna().sum()

amount_outlier_ratio

np.float64(0.10729791192888154)

Около 10.7% корректно распознанных значений Amount находятся за границами IQR. Это достаточно заметная доля, поэтому потенциальные выбросы нельзя просто автоматически удалить: сначала необходимо определить, отражают ли они реальные крупные транзакции или ошибки данных.

### Вывод по разделу 5

По рассчитанным метрикам датасет в среднем заполнен примерно на 96%, а доля полных дубликатов около 0.0165%.

При этом в `Amount` и `Balance` около 80% значений корректно распознаются как числовые, поэтому эти поля требуют дополнительной обработки. В невалидные ~20% попадают и «грязные» строки, и настоящие пропуски. На следующем шаге очистки эти два случая стоит обрабатывать по-разному. 

Для `Amount` доля потенциальных выбросов по методу IQR составляет около 10.7%.

Основные проблемы качества связаны не с общей полнотой данных, а с форматом и корректностью отдельных признаков.

## Общий вывод

Проведённый EDA показал, что датасет в целом достаточно полный и содержит минимальное количество полных дубликатов, однако отдельные признаки требуют дополнительной подготовки перед дальнейшим использованием.

Основные проблемы связаны с неоднородными форматами и некорректными значениями: часть числовых полей содержит посторонние символы, даты представлены в разных форматах, встречаются некорректные категории и неподходящие типы данных для идентификаторов. Для `Amount` также выявлена заметная доля потенциальных выбросов.

Перед дальнейшей обработкой необходимо привести типы и форматы данных к единому виду, обработать пропуски и некорректные значения, нормализовать отдельные категории и определить правила работы с повторяющимися идентификаторами и потенциальными выбросами.